# SDE-Seg — compact pilot
A short, reproducible experiment requested by the user: 3,000 training samples, 1,000 dev samples, 1,000 samples per test dataset, at most eight epochs per run, and 30 explanation samples. Four lambda candidates and three variants with three seeds are retained. Results are labelled **pilot_subset** and are not full-dataset benchmark results. Run cells in order in a GPU runtime.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
PROJECT = DRIVE/'sde-seg-run/research-v2'
RUN = PROJECT/'quick-pilot'
WORK = Path('/content/sde-seg')
SOURCE = PROJECT/'artifacts/sde-seg-source-6514cde77fd2d70dcc8446b7f1f6609a72a8332e2fe301e8039469b8adb6c6dc.zip'
LA_ZIP = DRIVE/'LA.zip'
LA_ROOT = WORK/'data/raw/LA'
ITW_MANIFEST = WORK/'data/manifests/itw.csv'
MLITW_MANIFEST = WORK/'data/manifests/mlitw.csv'
RUN.mkdir(parents=True, exist_ok=True)
def run(*arguments):
    subprocess.run([str(a) for a in arguments], check=True)


## 1. Install the recorded source
The active run must finish before another copy of this notebook changes source files.


In [ ]:
job_path = RUN/'logs/pilot_job.json'
if job_path.exists():
    job = json.loads(job_path.read_text())
    process = Path('/proc')/str(job.get('pid'))/'cmdline'
    if process.exists() and b'sdeseg.pilot' in process.read_bytes():
        raise RuntimeError('The compact pilot is already running; view its progress instead of starting another job.')
assert SOURCE.is_file(), f'Missing recorded source bundle: {SOURCE}'
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == SOURCE.stem.removeprefix('sde-seg-source-')
with zipfile.ZipFile(SOURCE) as archive:
    root = Path('/content').resolve()
    for name in archive.namelist():
        assert (root/name).resolve().is_relative_to(root)
    archive.extractall(root)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
os.environ['PYTHONPATH'] = str(WORK)
run(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-data.txt')
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'


## 2. Prepare the real source datasets
Existing extracted audio and manifests are reused. LA.zip is supplied by the user. Cross-domain downloads use pinned public sources and verify actual checksums. Their complete metadata is inspected, but the pilot only trains and scores on its fixed small subsets.


In [ ]:
if not (LA_ROOT/'.extraction_complete.json').exists():
    run(sys.executable, 'scripts/prepare_la.py', '--archive', LA_ZIP,
        '--data-root', LA_ROOT, '--results', PROJECT)
if not (ITW_MANIFEST.is_file() and MLITW_MANIFEST.is_file()):
    run(sys.executable, 'scripts/audit_sources.py', '--results', PROJECT)
    run(sys.executable, 'scripts/prepare_cross.py', '--data-root', WORK/'data/raw', '--results', PROJECT)
for name in ('aasist_parameters.json', 'dataset_sources.json', 'colab_environment.json'):
    source = PROJECT/'logs'/name
    if source.exists():
        (RUN/'logs').mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, RUN/'logs'/name)


## 3. Train and evaluate the compact experiment
The configuration is in `configs/pilot.yaml`. Sampling approximately preserves class proportions and uses the same subsets across all variants/seeds. All model choices are frozen before final-test subsets are opened. Matching completed evaluations are reused. Interrupted final scoring requires explicit review of its journal.


In [ ]:
run(sys.executable, '-u', '-m', 'sdeseg.pilot',
    '--data-root', LA_ROOT, '--subset-root', WORK/'data/pilot-la', '--results', RUN,
    '--itw-manifest', ITW_MANIFEST, '--mlitw-manifest', MLITW_MANIFEST,
    '--config', WORK/'configs/pilot.yaml', '--workers', '2')


## 4. View and save the measured results
The archive contains model checkpoints, logs, predictions, figures and tables; no raw dataset audio is included. Pilot scores must not be presented as full-dataset benchmark scores.


In [ ]:
import pandas as pd
from IPython.display import Markdown, display

job = json.loads((RUN/'logs/pilot_job.json').read_text())
assert job['stage'] == 'complete', 'Inspect pilot.log before treating the experiment as complete'
for name in ('table3_parameters.csv', 'table4_eer.csv', 'windowed_eer.csv', 'latency.csv'):
    print(name)
    display(pd.read_csv(RUN/'tables'/name))
display(Markdown((RUN/'REPORT.md').read_text()))
archive = shutil.make_archive(str(PROJECT/'SDE-Seg-pilot-results'), 'zip', root_dir=RUN)
print('Saved results archive:', archive)
print('Checkpoints, figures, predictions and report:', RUN)
